# Predicting Term Deposit Subscription with Ensemble Models
## Supervised Learning — Binary Classification in Bank Telemarketing

**Domain:** Banking / Direct Marketing  
**Target:** `y` — Term deposit subscription (`yes` / `no`)  
**Primary Paradigm:** Pre-Contact Lead Prioritization (Strictly excluding call `duration` to prevent target leakage)  
**Validation Strategy:** 80/20 Chronological Temporal Holdout + 5-fold TimeSeriesSplit


### 1. Environment Setup & Dependency Imports
We import standard scientific libraries: pandas, numpy, matplotlib, seaborn, scikit-learn, and joblib.

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score,
    roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

# Add root to sys.path to access local preprocessing modules
sys.path.insert(0, os.path.abspath('..'))
from src.preprocessing import build_full_pipeline, PreContactFeatureEngineer

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries successfully loaded.')

### 2. Dataset Loading & Data Quality Audit
We load `Bank_Marketing.csv` and inspect schema, data types, missing values, duplicates, and column semantics.

In [2]:
csv_path = '../Bank_Marketing.csv' if os.path.exists('../Bank_Marketing.csv') else '../dataset/Bank_Marketing.csv'
df = pd.read_csv(csv_path)
print('Dataset Shape:', df.shape)
print('Duplicate Rows:', df.duplicated().sum())
print('\nTarget Distribution:')
print(df['y'].value_counts())
print(df['y'].value_counts(normalize=True).apply(lambda x: f'{x*100:.2f}%'))

print('\nMissing Values Summary:')
missing = df.isnull().sum()
print(missing[missing > 0])

### 3. Exploratory Data Analysis (EDA)
Let us inspect the distribution of target variable, numeric features, and key categorical variables.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['y'].value_counts().plot(kind='bar', ax=axes[0], color=['#3498DB', '#2ECC71'], edgecolor='black')
axes[0].set_title('Class Distribution (Imbalance Ratio ~ 7.5 : 1)')
axes[0].set_ylabel('Number of Observations')

# Rolling subscription rate across sequential rows
rolling_sub = (df['y'] == 'yes').rolling(1000).mean() * 100
axes[1].plot(rolling_sub, color='#8E44AD', linewidth=1.5)
axes[1].axvline(int(len(df)*0.8), color='red', linestyle='--', label='80% Temporal Split')
axes[1].set_title('Rolling Subscription Rate Across Dataset Rows')
axes[1].set_xlabel('Chronological Row Index')
axes[1].set_ylabel('Subscription Rate (%)')
axes[1].legend()
plt.tight_layout()
plt.show()

### 4. Chronological Temporal Data Splitting
Because the dataset is sorted chronologically from May 2008 to November 2010, shuffling destroys the temporal arrow of time.
We use the first 80% (36,168 rows) as training data, and the final 20% (9,043 rows) as temporal holdout test data.

In [4]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df.drop(columns=['y'])
y_train = (train_df['y'] == 'yes').astype(int)
X_test = test_df.drop(columns=['y'])
y_test = (test_df['y'] == 'yes').astype(int)

print(f'Training set: {len(X_train):,} rows | Target positive rate: {y_train.mean()*100:.2f}%')
print(f'Holdout test set: {len(X_test):,} rows | Target positive rate: {y_test.mean()*100:.2f}%')

### 5. Preprocessing & Feature Engineering Pipeline
To prevent target leakage:
1. `duration` is strictly dropped via `PreContactFeatureEngineer`.
2. `previously_contacted` binary indicator is derived from `pdays != -1`.
3. Categorical missing values are filled with `Unknown` and one-hot encoded.
4. Numeric values are imputed with median.

In [5]:
from src.preprocessing import build_full_pipeline
sample_pipe = build_full_pipeline(LogisticRegression(class_weight='balanced'))
print('Pipeline Architecture:')
print(sample_pipe)

### 6. Model Training & Evaluation on Temporal Holdout
We load the trained best model and compare performance across models.

In [6]:
with open('../models/metrics_summary.json') as f:
    metrics_summary = json.load(f)

eval_rows = []
for name, res in metrics_summary['test_evaluations'].items():
    d = res['default_0.50']
    eval_rows.append({
        'Model': name,
        'PR-AUC': d['pr_auc'],
        'ROC-AUC': d['roc_auc'],
        'Balanced Acc': d['balanced_accuracy'],
        'F1-Score': d['f1'],
        'Recall': d['recall'],
        'Precision': d['precision'],
        'Specificity': d['specificity']
    })
eval_df = pd.DataFrame(eval_rows).set_index('Model')
print(eval_df.round(4).to_string())

### 7. Benchmarks: Target Leakage and Random Split
We compare the operational pre-contact model against:
1. **Post-Contact Leakage Model (with `duration`):** Demonstrates how knowing call length inflates test metrics artificially.
2. **Random Split Benchmark:** Demonstrates how random splitting masks temporal distribution shifts.

In [7]:
bm = metrics_summary['benchmarks']
print('Benchmark A: Post-Contact Leakage Model (WITH duration):')
print(pd.Series(bm['post_contact_model']['metrics']).round(4))
print('\nBenchmark B: Random-Split Benchmark (NO duration, Stratified Random Split):')
print(pd.Series(bm['random_split_model']['metrics']).round(4))

### 8. Permutation Feature Importance
We inspect the most impactful features on the temporal holdout test set.

In [8]:
with open('../models/permutation_importance.json') as f:
    perm_imp = pd.DataFrame(json.load(f))
print(perm_imp.sort_values(by='Importance_Mean', ascending=False).head(10).to_string(index=False))

### 9. Conclusion & Key Takeaways
- **Extra Trees & Random Forest** provide the strongest pre-contact discrimination, with PR-AUC ~ 0.491 and Balanced Accuracy ~ 0.643.
- Excluding `duration` ensures zero target leakage, making the model actionable for outbound call center prioritization.
- Temporal holdout validation successfully caught real-world concept drift where conversion rates jumped from 6.73% to 31.58% due to increased prior contact re-engagement.